# [v1 Retrospective] Feature Engineering
> **What this notebook shows:** Baseline feature preparation, column type identification, categorical encoding setup, and train/val splitting.
>
> **What went wrong in v1:**
> 1. **Fatal Leakage:** Used X = train_df.drop(columns=['label', 'attack_cat']), leaving id inside feature matrix X. Models split directly on row indices.
> 2. Random stratified splitting fragmented contiguous burst traffic between train and validation sets (data snooping).
> 3. Missed critical ratio features (e.g. bytes/packet, src/dst packet balance) and log transforms for heavy-tailed counters.

In [3]:
import pandas as pd
import numpy as np

In [4]:
train_df = pd.read_csv(
    "../../data/processed/selected_training.csv"
)

test_df = pd.read_csv(
    "../../data/processed/selected_testing.csv"
)


## Separating input features and target

In [5]:
X = train_df.drop(columns=["label", "attack_cat"])

y = train_df["attack_cat"].fillna("Normal")

## Identify numerical and categorical features

In [6]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']
Categorical: ['proto', 'service', 'state']


## Split the training data

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (98869, 43)
Validation: (24718, 43)


## Build the preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Even though you currently have no missing values, the imputers make the pipeline more robust. 

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])